# Stage 1 - Non-Instruction Fine-Tuning
### GenAI / Agentic AI Domain Assistant &middot; Base model: `Qwen2.5-0.5B` &middot; Framework: [Unsloth](https://github.com/unslothai/unsloth)

Pipeline: **Base Model &rarr; Stage 1: Non-Instruction FT &rarr; Stage 2: Instruction FT &rarr; Stage 3: DPO/ORPO Alignment**

This notebook:
1. Loads the pristine base model and evaluates it on 10 domain questions &rarr; writes `reports/base_model_evaluation.md`
2. Cleans/chunks the raw domain corpus (`data/non_instruction_data.txt`)
3. Runs a small non-instruction (continued-pretraining / raw-text) LoRA fine-tune to adapt the base model to GenAI/Agentic-AI vocabulary and writing style *before* any instruction tuning
4. Saves the resulting adapter to `models/non_instruction_adapter/`

> Run on a free Colab/Kaggle T4 GPU. Total runtime: ~5-10 minutes for this 0.5B model.


In [ ]:
%%capture
# Colab: Runtime > Change runtime type > T4 GPU, then run this cell first.
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U "trl>=0.12.0" peft accelerate bitsandbytes datasets sentencepiece


In [ ]:
import os

# If you're running this notebook straight from your cloned GitHub repo in
# Colab (File > Open notebook > GitHub), the paths below already work as-is.
# Otherwise, set REPO_URL and this cell will clone it for you.
REPO_URL = "https://github.com/Abhishek15016/prepminds.git"

if not os.path.exists("data") and REPO_URL:
    repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    if not os.path.exists(repo_name):
        get_ipython().system(f"git clone {REPO_URL}")
    os.chdir(repo_name)

assert os.path.exists("data"), "Could not find data/ - clone/open the repo first."
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
import sys
sys.path.insert(0, os.getcwd())


In [ ]:
from src.eval_questions import EVAL_QUESTIONS, clean_completion, judge_base_answer, judge_pair, markdown_table

print(f"Loaded {len(EVAL_QUESTIONS)} canonical evaluation questions.")
for i, q in enumerate(EVAL_QUESTIONS, 1):
    print(f"{i:2d}. {q}")


## Step 0 - (Optional) Hugging Face Hub setup

Enables persisting this stage's adapter to the Hub so the next notebook (a fresh Colab session) can load it back automatically instead of relying on local disk, which Colab wipes between sessions.

In [ ]:
from huggingface_hub import login
import os

# --- Hugging Face Hub config -------------------------------------------------
# Colab wipes local disk between sessions, so each notebook can optionally push
# its adapter to the Hub, and the *next* notebook will automatically pull it
# back down if no local copy is found. Set your HF username below and
# (recommended) add a Colab secret named HF_TOKEN with a "write" token -
# the key icon in the left sidebar - so login() doesn't need to prompt you.
HF_USERNAME = "abhishek15016"  # leave blank to skip the Hub entirely
PUSH_TO_HUB = bool(HF_USERNAME)
PRIVATE_REPO = True
HF_MODEL_PREFIX = "qwen2.5-0.5b-genai-agentic"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if PUSH_TO_HUB:
    login(token=HF_TOKEN or None)  # prompts interactively if HF_TOKEN isn't set

STAGE1_HF_REPO = f"{HF_USERNAME}/{HF_MODEL_PREFIX}-stage1-non-instruction" if HF_USERNAME else ""
STAGE2_HF_REPO = f"{HF_USERNAME}/{HF_MODEL_PREFIX}-stage2-sft" if HF_USERNAME else ""
STAGE3_HF_REPO = f"{HF_USERNAME}/{HF_MODEL_PREFIX}-stage3-dpo" if HF_USERNAME else ""

def push_adapter(model, tokenizer, repo_id, stage_name):
    if not (PUSH_TO_HUB and repo_id):
        print(f"Skipping Hub push for {stage_name} (set HF_USERNAME above to enable).")
        return
    model.push_to_hub(repo_id, token=HF_TOKEN or None, private=PRIVATE_REPO)
    tokenizer.push_to_hub(repo_id, token=HF_TOKEN or None, private=PRIVATE_REPO)
    print(f"Pushed {stage_name} adapter to https://huggingface.co/{repo_id}")


## Step A - Load the pristine base model and evaluate it (Step 5 of the assignment)

We test the **untouched** base model first, before any fine-tuning, so the before/after comparison later is honest.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
BASE_MODEL_NAME = "unsloth/Qwen2.5-0.5B-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect bf16/fp16
    load_in_4bit=True,   # QLoRA-style 4-bit loading
)


In [ ]:
FastLanguageModel.for_inference(model)  # Unsloth's 2x-faster inference path

def generate_raw(question, max_new_tokens=180):
    prompt = f"Question: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return clean_completion(text, question)

base_answers = [generate_raw(q) for q in EVAL_QUESTIONS]
for q, a in zip(EVAL_QUESTIONS, base_answers):
    print("Q:", q)
    print("A:", a[:300])
    print("-" * 80)


In [ ]:
rows = []
for q, a in zip(EVAL_QUESTIONS, base_answers):
    problem = judge_base_answer(q, a)
    rows.append((q, a, problem))

table = markdown_table(["Question", "Base Model Answer", "Problem"], rows)

report = f"""# Base Model Evaluation

**Model:** `{BASE_MODEL_NAME}` (pristine, no fine-tuning)
**Domain:** GenAI / Agentic AI interview-readiness assistant
**Purpose:** Establish the "before" baseline prior to Stage 1/2/3 fine-tuning (Assignment Step 5).

{table}

## Summary

The base model has broad general-purpose knowledge but no exposure to this
project's target domain style: interview-ready, structured, analogy-driven
explanations of GenAI/Agentic-AI concepts. Answers above are typically
generic, unstructured, or drift off-topic because the model is being
prompted with a plain completion format it wasn't instruction-tuned for.
This motivates Stage 1 (domain-adaptive non-instruction fine-tuning),
Stage 2 (instruction fine-tuning on Q&A pairs), and Stage 3 (DPO alignment)
carried out in this repository.

*Auto-generated by `notebooks/non_instruction_finetuning.ipynb`. The
"Problem" column is a heuristic first draft (see `src/eval_questions.py`) -
skim it and edit any row before submission if a note doesn't fit the
generated answer.*
"""

with open("reports/base_model_evaluation.md", "w", encoding="utf-8") as f:
    f.write(report)

print(report)


## Step B - Prepare the non-instruction (raw-text) dataset

`data/non_instruction_data.txt` holds ~50 raw domain explainer passages separated by `====...` rules.
We clean and chunk it into training examples, and cross-check against the already-prepared
`data/non_instruction_dataset.jsonl`.

In [ ]:
import re, json

with open("data/non_instruction_data.txt", encoding="utf-8") as f:
    raw = f.read()

# Clean + chunk: split on the "====" rule, strip whitespace, drop empty/too-short chunks.
chunks = [c.strip() for c in re.split(r"\n=+\n", raw)]
chunks = [c for c in chunks if len(c.split()) >= 15]
print(f"Chunked raw corpus into {len(chunks)} training passages.")

with open("data/non_instruction_dataset.jsonl", encoding="utf-8") as f:
    prebuilt = [json.loads(l) for l in f if l.strip()]
print(f"Pre-built data/non_instruction_dataset.jsonl has {len(prebuilt)} examples.")

from datasets import Dataset
raw_dataset = Dataset.from_list(prebuilt)  # use the canonical, already-verified file
raw_dataset


## Step C - Apply LoRA (QLoRA: 4-bit base + LoRA adapters) and train on raw text

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()


In [ ]:
from trl import SFTTrainer, SFTConfig
from src.trl_compat import build_config

sft_config = build_config(
    SFTConfig,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    padding_free=False,  # newer TRL auto-enables padding-free (flash-attn) packing, which conflicts with max_seq_length here
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs/stage1_non_instruction",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=raw_dataset,
    args=sft_config,
)


In [ ]:
# Take a screenshot of this cell's output for your README ("Training screenshots or logs").
train_result = trainer.train()
train_result.metrics


## Step D - Save the Stage 1 adapter

In [ ]:
SAVE_DIR = "models/non_instruction_adapter"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved Stage 1 (non-instruction) adapter to", SAVE_DIR)


In [ ]:
push_adapter(model, tokenizer, STAGE1_HF_REPO, "Stage 1 (non-instruction)")


## Step E - Sanity-check the model after Stage 1

Qualitative spot-check only (not a formal deliverable) - confirms the domain-adaptation signal took before moving on to instruction tuning.

In [ ]:
FastLanguageModel.for_inference(model)
for q in EVAL_QUESTIONS[:3]:
    print("Q:", q)
    print("A (post-Stage 1):", generate_raw(q)[:300])
    print("-" * 80)


---
### Next: `notebooks/instruction_finetuning.ipynb`

That notebook loads `models/non_instruction_adapter/` as its starting point (or,
if this ran in a separate Colab session, pulls it from `STAGE1_HF_REPO` on the
Hugging Face Hub instead) and performs instruction (Q&A) fine-tuning with LoRA
on `data/instruction_dataset.jsonl`.